# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DipeshGhimire33/Flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Plain-language rule:**

Prioritize pages for human review when they show a combination of **recent performance decline, older content, meaningful search visibility, and potential ranking/CTR opportunity**. The rule identifies pages worth investigating; it does not guarantee that refreshing them will improve performance.

**Reason codes:**

* **DECLINING_PERFORMANCE** — Recent impressions or clicks are lower than the previous 30-day period.
* **STALE_CONTENT** — The page has not been updated for a relatively long period.
* **HIGH_VISIBILITY** — The page receives meaningful search impressions, so a refresh could affect an established source of traffic.
* **RANKING_OPPORTUNITY** — The page has an average search position where optimization may be worth investigating.
* **CTR_OPPORTUNITY** — The page receives impressions but has relatively low CTR.
* **LOW_DATA_CONFIDENCE** — Important signals are missing or have limited historical data, so the page should be interpreted cautiously.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# -----------------------------
# 1. Build scoring components
# -----------------------------

# Performance change: recent vs previous 30 days
impression_change = (
    (df["impressions_prev_30d"] - df["impressions_last_30d"])
    / df["impressions_prev_30d"].replace(0, np.nan)
)

click_change = (
    (df["clicks_prev_30d"] - df["clicks_last_30d"])
    / df["clicks_prev_30d"].replace(0, np.nan)
)

# Convert to 0-1 opportunity signals
decline_signal = (
    (impression_change.clip(lower=0) +
     click_change.clip(lower=0)) / 2
)

# Freshness: older pages receive a higher opportunity signal
freshness_signal = (
    df["days_since_last_update"]
    .fillna(df["days_since_last_update"].median())
)

freshness_signal = (
    freshness_signal /
    freshness_signal.quantile(0.95)
).clip(0, 1)

# Visibility: pages with established impressions
visibility_signal = np.log1p(
    df["impressions_90d"].fillna(0)
)

visibility_signal = (
    visibility_signal /
    visibility_signal.quantile(0.95)
).clip(0, 1)

# Ranking opportunity: positions around page 1-2
position = df["avg_position"].replace(0, np.nan)

position_signal = (
    (position - 1) / 49
).clip(0, 1).fillna(0)

# CTR opportunity: lower CTR among visible pages
ctr_signal = (
    1 - df["ctr"].fillna(df["ctr"].median()) / 100
).clip(0, 1)

# -----------------------------
# 2. Combine into score
# -----------------------------

df["opportunity_score"] = (
    0.30 * decline_signal.fillna(0)
    + 0.20 * freshness_signal
    + 0.20 * visibility_signal
    + 0.15 * position_signal
    + 0.15 * ctr_signal
)

# -----------------------------
# 3. Add reason codes
# -----------------------------

def get_reasons(row):
    reasons = []

    if row["impressions_last_30d"] < row["impressions_prev_30d"]:
        reasons.append("DECLINING_PERFORMANCE")

    if row["days_since_last_update"] >= 180:
        reasons.append("STALE_CONTENT")

    if row["impressions_90d"] > df["impressions_90d"].median():
        reasons.append("HIGH_VISIBILITY")

    if 2 <= row["avg_position"] <= 20:
        reasons.append("RANKING_OPPORTUNITY")

    if row["ctr"] < df["ctr"].median():
        reasons.append("CTR_OPPORTUNITY")

    return "|".join(reasons)

df["reason_codes"] = df.apply(get_reasons, axis=1)

# -----------------------------
# 4. Rank the queue
# -----------------------------

df = df.sort_values(
    "opportunity_score",
    ascending=False
).reset_index(drop=True)

df["priority_rank"] = df.index + 1

# -----------------------------
# 5. Save output
# -----------------------------

output_cols = [
    "content_id",
    "opportunity_score",
    "priority_rank",
    "reason_codes"
]

output = df[output_cols]

output.to_csv(
    "../../work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", "../../work/outputs/baseline_action_score.csv")
print("Rows:", len(output))
print(output.head(10))

Saved: ../../work/outputs/baseline_action_score.csv
Rows: 30000
             content_id  opportunity_score  priority_rank  \
0  content_fb66dd8f4629           0.977913              1   
1  content_9623e642ff3a           0.950456              2   
2  content_045de7f327fe           0.944312              3   
3  content_058efde65398           0.927207              4   
4  content_3dbf0b8caf06           0.926736              5   
5  content_061b51236cbd           0.922633              6   
6  content_8b1eeb87a7c5           0.922307              7   
7  content_581f85ef23d3           0.921521              8   
8  content_bba7a7a249b2           0.921460              9   
9  content_9b8a4ceacea5           0.917693             10   

                                        reason_codes  
0              DECLINING_PERFORMANCE|HIGH_VISIBILITY  
1  DECLINING_PERFORMANCE|HIGH_VISIBILITY|CTR_OPPO...  
2  DECLINING_PERFORMANCE|HIGH_VISIBILITY|CTR_OPPO...  
3  DECLINING_PERFORMANCE|HIGH_VISIBILITY|CTR

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
df = pd.read_csv("../../work/outputs/baseline_action_score.csv")

top20 = df.head(20).copy()

# Basic review fields
top20["action"] = "REVIEW_FOR_REFRESH"

def confidence_note(row):
    if not row["reason_codes"]:
        return "Low confidence: limited supporting signals."
    return "Medium confidence: multiple signals support review."

def what_could_make_it_wrong(row):
    return (
        "Missing or unreliable data, temporary traffic changes, "
        "or business context could make refresh unnecessary."
    )

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(
    what_could_make_it_wrong, axis=1
)

review_cols = [
    "content_id",
    "priority_rank",
    "opportunity_score",
    "action",
    "reason_codes",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_cols]

top20_review
# print(top20_review.to_string(index=False))

,content_id,priority_rank,opportunity_score,action,reason_codes,confidence_note,what_would_make_it_wrong
0,content_fb66dd8f4629,1,0.977913,REVIEW_FOR_REFRESH,DECLINING_PERFORMANCE|HIGH_VISIBILITY,Medium confidence: multiple signals support re...,"Missing or unreliable data, temporary traffic ..."
1,content_9623e642ff3a,2,0.950456,REVIEW_FOR_REFRESH,DECLINING_PERFORMANCE|HIGH_VISIBILITY|CTR_OPPO...,Medium confidence: multiple signals support re...,"Missing or unreliable data, temporary traffic ..."
2,content_045de7f327fe,3,0.944312,REVIEW_FOR_REFRESH,DECLINING_PERFORMANCE|HIGH_VISIBILITY|CTR_OPPO...,Medium confidence: multiple signals support re...,"Missing or unreliable data, temporary traffic ..."
3,content_058efde65398,4,0.927207,REVIEW_FOR_REFRESH,DECLINING_PERFORMANCE|HIGH_VISIBILITY|CTR_OPPO...,Medium confidence: multiple signals support re...,"Missing or unreliable data, temporary traffic ..."
4,content_3dbf0b8caf06,5,0.926736,REVIEW_FOR_REFRESH,DECLINING_PERFORMANCE|HIGH_VISIBILITY,Medium confidence: multiple signals support re...,"Missing or unreliable data, temporary traffic ..."
5,content_061b51236cbd,6,0.922633,REVIEW_FOR_REFRESH,DECLINING_PERFORMANCE|HIGH_VISIBILITY|CTR_OPPO...,Medium confidence: multiple signals support re...,"Missing or unreliable data, temporary traffic ..."
6,content_8b1eeb87a7c5,7,0.922307,REVIEW_FOR_REFRESH,DECLINING_PERFORMANCE|HIGH_VISIBILITY|CTR_OPPO...,Medium confidence: multiple signals support re...,"Missing or unreliable data, temporary traffic ..."
7,content_581f85ef23d3,8,0.921521,REVIEW_FOR_REFRESH,DECLINING_PERFORMANCE|HIGH_VISIBILITY,Medium confidence: multiple signals support re...,"Missing or unreliable data, temporary traffic ..."
8,content_bba7a7a249b2,9,0.921460,REVIEW_FOR_REFRESH,DECLINING_PERFORMANCE|HIGH_VISIBILITY|CTR_OPPO...,Medium confidence: multiple signals support re...,"Missing or unreliable data, temporary traffic ..."
9,content_9b8a4ceacea5,10,0.917693,REVIEW_FOR_REFRESH,DECLINING_PERFORMANCE|HIGH_VISIBILITY,Medium confidence: multiple signals support re...,"Missing or unreliable data, temporary traffic ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# Check the lowest-scoring pages among the review queue
weak_picks = df.tail(20).copy()

print(weak_picks[
    ["content_id", "opportunity_score", "reason_codes"]
].to_string(index=False))


# -----------------------------
# Leakage / exclusion check
# -----------------------------

feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "search_volume",
    "avg_position",
    "ctr"
]


excluded = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "trend_direction",
    "trend_pct"
]

future_like = [
    col for col in feature_cols
    if any(word in col.lower()
           for word in ["future", "next", "forecast"])
]

used_excluded = [
    col for col in feature_cols
    if col in excluded
]

print("\nExcluded fields accidentally used:", used_excluded)
print("Potential future-window features:", future_like)

          content_id  opportunity_score        reason_codes
content_eb47eae80392           0.136870                 NaN
content_841cd12bb30f           0.135339                 NaN
content_7debdb0d3a8a           0.135339                 NaN
content_0f96bf7b0be4           0.135339                 NaN
content_76b07f20b83c           0.135339                 NaN
content_78467f5a1702           0.135339                 NaN
content_1e826070c703           0.135339                 NaN
content_f03ce106b8b7           0.135339                 NaN
content_3a16f4fb4fc6           0.135339                 NaN
content_b96873ca64c1           0.126038 RANKING_OPPORTUNITY
content_4e95a8389562           0.124507 RANKING_OPPORTUNITY
content_f26233911f33           0.115324 RANKING_OPPORTUNITY
content_cfa4d9f1bf0a           0.102657                 NaN
content_a84e013a5f94           0.076755 RANKING_OPPORTUNITY
content_bf398aa7400e           0.055326 RANKING_OPPORTUNITY
content_b1e4f7904d85           0.052265 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.